In [1]:
import spacy
print(spacy.__version__)

3.7.2


In [6]:
import spacy
from spacy import displacy

def find_main_verb(doc):
    """
    Tìm token có quan hệ phụ thuộc là ROOT (động từ chính) trong đối tượng Doc của spaCy.
    Args:
        doc (spacy.tokens.Doc): Đối tượng Doc đã được xử lý bởi spaCy.
    Returns:
        spacy.tokens.Token or None: Token là động từ chính (ROOT), hoặc None nếu không tìm thấy.
    """
    for token in doc:
        if token.dep_ == "ROOT":
            return token
    return None

nlp = spacy.load("en_core_web_sm")
text = "The quick brown fox jumps over the lazy dog."
doc = nlp(text)
# displacy.serve(doc, style="dep")

main_verb = find_main_verb(doc)
if main_verb:
    print(f"Động từ chính: {main_verb.text} ({main_verb.pos_})")
else:
    print("Không tìm thấy động từ chính.")

Động từ chính: jumps (VERB)


In [5]:
import spacy

def custom_noun_chunks(doc):
    """
    Trích xuất các cụm danh từ đơn giản từ đối tượng Doc.
    Phương pháp đơn giản: Bắt đầu từ một danh từ chính (NOUN/PROPN),
    và thêm tất cả các children bổ nghĩa trực tiếp cho nó.
    Args:
        doc (spacy.tokens.Doc): Đối tượng Doc đã được xử lý bởi spaCy.
    Returns:
        list: Danh sách các chuỗi (string) đại diện cho các cụm danh từ.
    """
    noun_chunks_list = []

    modifier_deps = {"det", "amod", "compound", "nummod", "poss", "quantmod", "predet"}
    used_tokens = set()

    for token in doc:
        if token.pos_ in ("NOUN", "PROPN") and token not in used_tokens:
            chunk_tokens = [token] # Bắt đầu với head

            # Thêm các children bổ nghĩa trực tiếp cho head
            for child in token.children:
                if child.dep_ in modifier_deps:
                    chunk_tokens.append(child)

            chunk_tokens.sort(key=lambda t: t.i)
            chunk_text = " ".join([t.text for t in chunk_tokens])
            noun_chunks_list.append(chunk_text)

            for t in chunk_tokens:
                used_tokens.add(t)

    return noun_chunks_list

nlp = spacy.load("en_core_web_sm")
text = "A quick red fox jumps over the lazy brown dog."
doc = nlp(text)

# Kết quả sử dụng thuộc tính có sẵn của spaCy để so sánh
print("spaCy built-in:", [chunk.text for chunk in doc.noun_chunks])

# Kết quả của hàm tự viết
custom_chunks = custom_noun_chunks(doc)
print("Custom function:", custom_chunks)

spaCy built-in: ['A quick red fox', 'the lazy brown dog']
Custom function: ['A quick red fox', 'the lazy brown dog']


In [1]:
import spacy

def get_path_to_root(token):
    """
    Tìm đường đi từ token hiện tại lên đến gốc (ROOT) của cây dependency.
    Args:
        token: Một đối tượng Token (của spaCy).
    Returns:
        list: Danh sách các token từ token bắt đầu đến gốc.
    """
    path = []
    current_token = token

    while True:
        # 1. Thêm token hiện tại vào đường đi
        path.append(current_token)

        # 2. Kiểm tra điều kiện dừng: Nếu token trỏ vào chính nó, đó là ROOT
        if current_token.head == current_token:
            break

        # 3. Di chuyển lên cha (head) của token hiện tại
        current_token = current_token.head

    return path

nlp = spacy.load("en_core_web_sm")
doc = nlp("Autonomous cars shift insurance liability toward manufacturers.")
target_token = doc[6]
path = get_path_to_root(target_token)

print(f"Đường đi từ '{target_token.text}' lên ROOT:")
for t in path:
    print(f" -> {t.text} ({t.dep_})")

Đường đi từ 'manufacturers' lên ROOT:
 -> manufacturers (pobj)
 -> toward (prep)
 -> shift (ROOT)
